## This is a simple example to run a GPU-enabled Flask API in Google Colab using the CuPy

### Instructions 

**Enable GPU**

1. Click Runtime → Change runtime type.

2. Set Hardware accelerator → GPU.

3. Click Save.

In [ ]:
import cupy as cp

if cp.cuda.is_available():
    device = cp.cuda.Device()  # default device 0
    props = cp.cuda.runtime.getDeviceProperties(device.id)
    device_name = props['name'].decode('utf-8')  # convert bytes to str
    print("GPU is available:", device_name)
else:
    print("GPU not available, using CPU")

In [ ]:
# Install required packages (run once in Colab)
!pip install flask  cupy-cuda12x requests

In [ ]:
# -------------------------
# Import necessary libraries
# -------------------------
from flask import Flask, jsonify  # Flask: lightweight web framework, jsonify: convert Python dicts to JSON
import cupy as cp                 # CuPy: GPU-accelerated array library (like NumPy but runs on GPU)
import threading                  # To run the Flask server in a background thread
import time                       # For adding short delays (used to ensure server starts)
import requests                   # For testing the API by sending HTTP requests

# -------------------------
# Create Flask application
# -------------------------
app = Flask(__name__)  # Initialize a Flask application object

# -------------------------
# Helper function: Get GPU device name
# -------------------------
def get_device_name():
    """
    Checks if a GPU is available and returns its name.
    If no GPU is available, returns "CPU".
    """
    if cp.cuda.is_available():           # Check if CuPy detects a CUDA-enabled GPU
        device = cp.cuda.Device()        # Get the current GPU device
        props = cp.cuda.runtime.getDeviceProperties(device.id)  # Get device properties
        return props["name"].decode("utf-8")  # Extract device name as string
    return "CPU"  # Fallback if no GPU is detected

# -------------------------
# Endpoint: Health check
# -------------------------
@app.route("/")  # URL path: /
def health():
    """
    Simple health check endpoint.
    Returns the status of the API and the device being used (CPU or GPU).
    """
    return {"status": "API running", "device": get_device_name()}

# -------------------------
# Endpoint: GPU matrix multiplication
# -------------------------
@app.route("/gpu-matrix")  # URL path: /gpu-matrix
def gpu_matrix():
    """
    Creates two large random matrices on the GPU and multiplies them.
    Returns the shape of the result and the device used.
    """
    # Generate two random 1000x1000 matrices on GPU
    x = cp.random.randn(1000, 1000)
    y = cp.random.randn(1000, 1000)

    # Matrix multiplication (runs on GPU)
    result = cp.matmul(x, y)

    # Return JSON response with shape and device info
    return {"shape": list(result.shape), "device": get_device_name()}

# -------------------------
# Endpoint: GPU random array
# -------------------------
@app.route("/gpu-random-array")  # URL path: /gpu-random-array
def gpu_random_array():
    """
    Creates a 500x500 random array on GPU and computes its sum.
    Returns the shape, sum, and device used.
    """
    arr = cp.random.rand(500, 500)      # Random array on GPU
    return {"shape": list(arr.shape), "sum": float(cp.sum(arr)), "device": get_device_name()}

# -------------------------
# Run Flask server in background
# -------------------------
def run_app():
    """
    Starts the Flask server on port 5000.
    Normally, app.run() blocks execution, but we will run it in a background thread
    so that we can interact with the API in the same notebook.
    """
    app.run(port=5000)

# Create a background thread to run Flask
thread = threading.Thread(target=run_app)
thread.daemon = True  # Daemon thread: automatically exits when main program exits
thread.start()        # Start the Flask server in the background



In [ ]:
# Call the endpoints
base_url = "http://127.0.0.1:5000"
print(requests.get(base_url + "/").json())
print(requests.get(base_url + "/gpu-matrix").json())
print(requests.get(base_url + "/gpu-random-array").json())

###  **Explanation of key points:**

**Flask endpoints:**

- / → simple health check

- /gpu-matrix → demonstrates a computationally heavy GPU operation

- /gpu-random-array → smaller GPU operation suitable for exercises

**Background thread:**

Normally app.run() blocks execution, so we run it in a daemon thread. This allows the same Colab notebook to send requests to the server while it’s running.

### Why can't you access (http://127.0.0.1:5000) from your local machine?

- When you run a Colab notebook, the Python process is actually running on a **Google server in the cloud, not on your local machine.**

- 127.0.0.1 (localhost) always refers to the same machine the code is running on.

- when you start Flask with app.run(port=5000), it binds to 127.0.0.1 inside the Colab VM, which your browser or another notebook cell **cannot access directly**.